In [242]:
import os
import sys
sys.path.append("..")  # add project root
from prog import mlps, featlib, trainer, hlprs
import Datasets.matconv as mc
import torch
import matplotlib.pyplot as plt

### Testing Symnet for Hardcoded Accuracy vs autodiff wrt $t$

    - Use MLP with activation to create regularized fit of data. 
    - Create Symnet object with hardcoded entries 
    - Use generated input tensor F to compare ut over same batch. 

In [10]:
device='cpu'

### Data Settings

In [263]:
# Data generation settings
noise=0.0
nu=0.02
stride_t=1
stride_x=1
part_num=1
which_part=1
seed=1432

In [264]:
# Dataset building

partitions = mc.build_dataset_from_burgers(noise_level=noise, nu=nu, stride_t=stride_t, stride_x=stride_x, seed=seed, quantile_splits=part_num, return_partitions=True)
t_np, x_np, y_np, y_noisy_np, N = partitions[[k for k in partitions if k.startswith(f'Q{which_part}:')][0]]
# batches are generated over t_np, x_np, y_np, y_noisy_np

t_torch       = torch.from_numpy(t_np).to(device)
x_torch       = torch.from_numpy(x_np).to(device)
y_torch       = torch.from_numpy(y_np).to(device)
y_noisy_torch = torch.from_numpy(y_noisy_np).to(device)

#### Training Settings

Loss is calculated like
$$ L[\tilde{u}] = \lambda_{data}||\tilde{u}-u_{raw}||_{L2}+\lambda_{pde}||\tilde{u}_t - Sym(\tilde{u},\tilde{u_x},\tilde{u_{xx}})||_{mse}+\lambda_{reg}||w_{symnet}||+\lambda_{tv}||\tilde{u}||_{tv}$$

In [283]:
selected_derivs=('u','u_x','u_xx') # up to first order derivative inputs
lr=0.001
batch_size=1000  # number of data points used on optimization step 
                 # data points are generated according to tensor.randint over spacetime mesh
lam_pde=0.0
lam_reg=0.0
lam_tv=0.0
lam_data=10.0

#output settings
log_every=1000

In [284]:
# Initializing u_model and a dummy v_model so that trainer works
u_model=mlps.SimpleMLP(n_layers=4,hidden_size=64)
v_model=mlps.EQL(in_dim=len(selected_derivs), prod_dim=0, bias=False) # dummy v_model for working code
for param in v_model.parameters(): #freeze v_model params
    param.requires_grad = False
print('v model parameters (weights) are not receiving gradients (improve data fit accuracy)')

train_config=trainer.TrainerConfig(
    lr=lr,
    lambda_pde=lam_pde, 
    lambda_reg=lam_reg,
    lambda_tv=lam_tv, 
    lambda_data=lam_data, 
    selected_derivs=selected_derivs, 
)
ft = featlib.FeatureTensor(selected_derivs, normalize=False)
feature_builder = ft.build
train=trainer.PDETrainer(u_model=u_model,v_model=v_model,cfg=train_config, feature_builder=feature_builder)
'rmrfjk'

v model parameters (weights) are not receiving gradients (improve data fit accuracy)


C:\Users\sami\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\nn\init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


'rmrfjk'

### Training Loop

In [285]:
print(list(train.v.parameters()))

[Parameter containing:
tensor([], size=(0, 3)), Parameter containing:
tensor([[-0.1850, -0.3297, -0.1513, -0.0121]])]


#### MLP gets trained over raw data

In [286]:
for i in range(4000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    if i%log_every==0:
        print(f'step {i} data_loss={ld} pde_loss={lp}')

step 0 data_loss=0.3880060613155365 pde_loss=0.003993150778114796
step 1000 data_loss=0.0030573178082704544 pde_loss=0.417607843875885
step 2000 data_loss=0.0002771335421130061 pde_loss=0.6951391100883484
step 3000 data_loss=9.716362546896562e-05 pde_loss=0.8496084213256836


### Test 2: Hardcoded Symnet weights and u_model pretrained on data

In [313]:
symnet = mlps.EQL(in_dim=3, prod_dim=2, bias=False)
symnet.linear.weight
with torch.no_grad():
    symnet.linear.weight.copy_(torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=symnet.linear.weight.dtype, device=symnet.linear.weight.device))
    symnet.readout.weight.copy_(torch.tensor([[0.0, 0, 0.02, -1.00]], dtype=symnet.readout.weight.dtype, device=symnet.readout.weight.device))
print("symnet hardcoded weights:")
print(symnet.readout.weight, '\n')
print(symnet.linear.weight)

symnet hardcoded weights:
Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0200, -1.0000]], requires_grad=True) 

Parameter containing:
tensor([[1., 0., 0.],
        [0., 1., 0.]], requires_grad=True)


We compare u_t over current batch with Symnet. This demonstrates the result of $$L_{pde} = \lambda_{pde}||u_t - Symnet||$$

In [314]:
# Check accuracy of hardcoded Symnet
symnet_error = train.mse(train.u_t,symnet(train.F))
print(f'SYMNET error vs u_t (autograd) with hardcoded parameters (forced selected pde): \n{symnet_error}')

SYMNET error vs u_t (autograd) with hardcoded parameters (forced selected pde): 
0.005061391741037369


In [315]:
train.v = symnet # swap out v_model for symnet
print(train.v.linear.weight) # verify weights are hardcoded
params = list(train.u.parameters()) + list(train.v.parameters())
train.optimizer = torch.optim.Adam(params, lr=lr) # update optimizer to replace dummy v model with symnet weights


Parameter containing:
tensor([[1., 0., 0.],
        [0., 1., 0.]], requires_grad=True)


In [316]:
# Freeze mlp for data fit, regularizer on data term to zero, loss only depends on pde loss 
# Compare loss values above 

train_config.lambda_data = 0.0
train_config.lambda_pde = 0.5
for params in u_model.parameters(): 
    params.requires_grad = False

for params in symnet.parameters(): 
    params.requires_grad = True

In [317]:
plot_pde_ls = []

for i in range(10000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    plot_pde_ls.append(lp)
    if i%log_every==0:
        print(f'step_{i}, data_loss={ld}, pde_loss={lp}')

step_0, data_loss=5.883095582248643e-05, pde_loss=0.004939001519232988
step_1000, data_loss=6.059840598027222e-05, pde_loss=0.002838448155671358
step_2000, data_loss=6.721723912050948e-05, pde_loss=0.0023544016294181347
step_3000, data_loss=5.7836419728118926e-05, pde_loss=0.002873615827411413
step_4000, data_loss=5.9374877309892327e-05, pde_loss=0.002858701627701521
step_5000, data_loss=6.442197627620772e-05, pde_loss=0.0026899699587374926
step_6000, data_loss=6.378135003615171e-05, pde_loss=0.0029301405884325504
step_7000, data_loss=5.898285598959774e-05, pde_loss=0.0023848023265600204
step_8000, data_loss=7.091084262356162e-05, pde_loss=0.0026551939081400633
step_9000, data_loss=6.622212822549045e-05, pde_loss=0.0033222909551113844


In [318]:
symnet.linear.weight, symnet.readout.weight

(Parameter containing:
 tensor([[1.0402, 0.0046, 0.0126],
         [0.0019, 0.9960, 0.0070]], requires_grad=True),
 Parameter containing:
 tensor([[-0.0091, -0.0035,  0.0037, -0.9639]], requires_grad=True))

Q: How do we know symnet is receiving gradients? 

### Test 3: random initialization of symnet and u_model pretrained on data

In [319]:
symnet = mlps.EQL(in_dim=3, prod_dim=2, bias=False)
symnet.linear.weight

Parameter containing:
tensor([[-0.1933,  0.3473,  0.2582],
        [-0.2700,  0.4469,  0.0030]], requires_grad=True)

In [320]:
train.v = symnet # swap out v_model for symnet
print(train.v.linear.weight) # verify weights are hardcoded
params = list(train.u.parameters()) + list(train.v.parameters())
train.optimizer = torch.optim.Adam(params, lr=lr) # update optimizer to replace dummy v model with symnet weights


Parameter containing:
tensor([[-0.1933,  0.3473,  0.2582],
        [-0.2700,  0.4469,  0.0030]], requires_grad=True)


In [321]:
#Sanity check: set loss to L_p, and freeze the data fit while training only symnet

train_config.lambda_data = 0.0
train_config.lambda_pde = 0.5
for params in u_model.parameters(): 
    params.requires_grad = False

for params in symnet.parameters(): 
    params.requires_grad = True

In [324]:
plot_pde_ls = []

for i in range(4000): 
    t,x,u_noisy,u_clean = hlprs.make_nonlinear_batch(batch_size=batch_size,t_torch=t_torch,x_torch=x_torch,y_torch=y_noisy_torch)
    out = train.step(t=t,x=x,u_noisy=u_noisy,u_clean=u_clean)
    ld = out['loss_data']
    lp = out['loss_pde']
    plot_pde_ls.append(lp)
    if i%log_every==0:
        print(f'step_{i}, data_loss={ld}, pde_loss={lp}')

step_0, data_loss=6.684457912342623e-05, pde_loss=0.0030670727137476206
step_1000, data_loss=6.81054443703033e-05, pde_loss=0.0027340988162904978
step_2000, data_loss=6.36560725979507e-05, pde_loss=0.0031218414660543203
step_3000, data_loss=6.0407310229493305e-05, pde_loss=0.0029431013390421867


In [325]:
print(list(train.v.parameters()))

[Parameter containing:
tensor([[ 0.0038,  1.3070,  0.0084],
        [-1.3215, -0.0047, -0.0162]], requires_grad=True), Parameter containing:
tensor([[-0.0108, -0.0046,  0.0033,  0.5782]], requires_grad=True)]
